# timesfm final — v3 champion raw-input pipeline registration

this notebook performs no training. it packages the validated v3 champion — 40% seasonal naive, 5% raw timesfm, 45% residual timesfm and 10% timesfm xreg — into a fitted object whose `predict(raw_test)` method owns all preprocessing. the object embeds train history, external features, store metadata, missing-value rules, covariate construction and the final blend. it is contract-tested on the full raw kaggle test set and linked to the w&b model registry.

In [48]:
# use the same official timesfm/xreg package as the validated v3 experiment.
%pip install -q "timesfm[torch,xreg]==2.0.2" "wandb==0.28.0" cloudpickle

In [49]:
import os

os.environ["JAX_PLATFORMS"] = "cpu"
os.environ["XLA_PYTHON_CLIENT_PREALLOCATE"] = "false"

import hashlib
import json
import platform
import time
from pathlib import Path

import cloudpickle
import numpy as np
import pandas as pd
import timesfm
import torch
import wandb

torch.set_float32_matmul_precision("high")
print(
    {
        "python": platform.python_version(),
        "torch": torch.__version__,
        "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu",
    }
)

{'python': '3.12.13', 'torch': '2.11.0+cu128', 'gpu': 'Tesla T4'}


In [50]:
config = {
    "data_dir": "/content/drive/MyDrive/walmart_competition_data",
    "output_dir": "/content/artifacts/timesfm_final_pipeline",
    "model_id": "google/timesfm-2.5-200m-pytorch",
    "source_evaluation_artifact": "timesfm-v3-xreg-corrected-calibration:latest",
    "registry_target": "wandb-registry-model/Walmart_TimesFM_Raw_Pipeline",
    "artifact_name": "timesfm-v3-raw-input-pipeline",
    "wandb_entity": "kende23-n-a",
    "wandb_project": "Walmart-Recruiting---Store-Sales-Forecasting",
    "wandb_run_name": "timesfm_v3_raw_pipeline_registration",
    "weights": {"seasonal": 0.40, "raw": 0.05, "residual": 0.45, "xreg": 0.10},
    "xreg_mode": "timesfm + xreg",
    "seasonal_period": 52,
    "clip_min": 0.0,
    "clip_max": 300000.0,
    "batch_size": 32,
}
data_dir = Path(config["data_dir"])
output_dir = Path(config["output_dir"])
output_dir.mkdir(parents=True, exist_ok=True)
print(config)

{'data_dir': '/content/drive/MyDrive/walmart_competition_data', 'output_dir': '/content/artifacts/timesfm_final_pipeline', 'model_id': 'google/timesfm-2.5-200m-pytorch', 'source_evaluation_artifact': 'timesfm-v3-xreg-corrected-calibration:latest', 'registry_target': 'wandb-registry-model/Walmart_TimesFM_Raw_Pipeline', 'artifact_name': 'timesfm-v3-raw-input-pipeline', 'wandb_entity': 'kende23-n-a', 'wandb_project': 'Walmart-Recruiting---Store-Sales-Forecasting', 'wandb_run_name': 'timesfm_v3_raw_pipeline_registration', 'weights': {'seasonal': 0.4, 'raw': 0.05, 'residual': 0.45, 'xreg': 0.1}, 'xreg_mode': 'timesfm + xreg', 'seasonal_period': 52, 'clip_min': 0.0, 'clip_max': 300000.0, 'batch_size': 32}


In [51]:
try:
    from google.colab import drive, userdata

    drive.mount("/content/drive")
    wandb_key = userdata.get("WANDB_API_KEY")
except Exception:
    wandb_key = None
wandb.login(key=wandb_key) if wandb_key else wandb.login()

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: WARNING [wandb.login()] Changing session credentials to explicit value for https://api.wandb.ai.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc


True

## load the fitted pipeline state

In [52]:
def locate_csv(root: Path, name: str) -> Path:
    for path in (root / name, root / f"{name}.zip"):
        if path.exists():
            return path
    raise FileNotFoundError(f"missing {name} or {name}.zip in {root}")


train_raw = pd.read_csv(locate_csv(data_dir, "train.csv"), parse_dates=["Date"])
test_raw = pd.read_csv(locate_csv(data_dir, "test.csv"), parse_dates=["Date"])
features_raw = pd.read_csv(locate_csv(data_dir, "features.csv"), parse_dates=["Date"])
stores_raw = pd.read_csv(locate_csv(data_dir, "stores.csv"))
train_raw = train_raw.sort_values(["Store", "Dept", "Date"]).reset_index(drop=True)
test_raw = test_raw.reset_index(drop=True)
assert not train_raw.duplicated(["Store", "Dept", "Date"]).any()
assert not test_raw.duplicated(["Store", "Dept", "Date"]).any()
print(
    {
        "train_rows": len(train_raw),
        "test_rows": len(test_raw),
        "train_series": train_raw.groupby(["Store", "Dept"]).ngroups,
        "test_series": test_raw.groupby(["Store", "Dept"]).ngroups,
    }
)

{'train_rows': 421570, 'test_rows': 115064, 'train_series': 3331, 'test_series': 3169}


## raw pipeline class

the pretrained checkpoint is identified by its immutable hugging face model id and loaded lazily on the first prediction. it is excluded from pickle state, keeping the registry artifact compact. all walmart-specific fitted state is embedded in the pipeline.

In [53]:
class TimesFMRawPipeline:
    required_columns = ("Store", "Dept", "Date", "IsHoliday")

    def __init__(
        self,
        history: pd.DataFrame,
        external_features: pd.DataFrame,
        stores: pd.DataFrame,
        model_id: str,
        weights: dict[str, float],
        xreg_mode: str = "timesfm + xreg",
        seasonal_period: int = 52,
        clip_min: float = 0.0,
        clip_max: float = 300000.0,
        batch_size: int = 32,
    ):
        self.history = history[
            ["Store", "Dept", "Date", "Weekly_Sales", "IsHoliday"]
        ].copy()
        self.history["Date"] = pd.to_datetime(self.history["Date"])
        self.external_features = external_features.copy()
        self.external_features["Date"] = pd.to_datetime(self.external_features["Date"])
        self.stores = stores.copy()
        self.model_id = model_id
        self.weights = dict(weights)
        self.xreg_mode = xreg_mode
        self.seasonal_period = int(seasonal_period)
        self.clip_min = float(clip_min)
        self.clip_max = float(clip_max)
        self.batch_size = int(batch_size)
        self._model = None
        self._validate_weights()

    def __getstate__(self):
        state = self.__dict__.copy()
        state["_model"] = None
        return state

    def _validate_weights(self) -> None:
        expected = {"seasonal", "raw", "residual", "xreg"}
        if set(self.weights) != expected:
            raise ValueError(f"weights must have keys {sorted(expected)}")
        if not np.isclose(sum(self.weights.values()), 1.0):
            raise ValueError("pipeline weights must sum to one")

    def metadata(self) -> dict:
        return {
            "pipeline_type": type(self).__name__,
            "model_id": self.model_id,
            "weights": self.weights,
            "xreg_mode": self.xreg_mode,
            "seasonal_period": self.seasonal_period,
            "stored_history_rows": len(self.history),
            "stored_feature_rows": len(self.external_features),
            "raw_input_columns": list(self.required_columns),
        }

    def _load_model(self):
        if self._model is None:
            if not torch.cuda.is_available():
                print("warning: timesfm pipeline is running without a gpu")
            model = timesfm.TimesFM_2p5_200M_torch.from_pretrained(self.model_id)
            model.compile(
                timesfm.ForecastConfig(
                    max_context=256,
                    max_horizon=64,
                    normalize_inputs=True,
                    per_core_batch_size=self.batch_size,
                    use_continuous_quantile_head=False,
                    force_flip_invariance=True,
                    infer_is_positive=True,
                    fix_quantile_crossing=True,
                    return_backcast=True,
                )
            )
            self._model = model
        return self._model

    @staticmethod
    def _event_name(date: pd.Timestamp, is_holiday: bool) -> str:
        if not is_holiday:
            return "none"
        return {
            2: "super_bowl",
            9: "labor_day",
            11: "thanksgiving",
            12: "christmas",
        }.get(date.month, "other_holiday")

    def _series_matrices(self, raw_test: pd.DataFrame):
        history_dates = pd.DatetimeIndex(sorted(self.history["Date"].unique()))
        forecast_dates = pd.DatetimeIndex(sorted(raw_test["Date"].unique()))
        keys = sorted(
            map(tuple, raw_test[["Store", "Dept"]].drop_duplicates().to_numpy())
        )
        lookup = self.history.set_index(["Store", "Dept", "Date"])["Weekly_Sales"]
        history_matrix = np.zeros((len(keys), len(history_dates)), dtype=np.float32)
        seasonal = np.zeros((len(keys), len(forecast_dates)), dtype=np.float32)
        for index, (store, dept) in enumerate(keys):
            history_index = pd.MultiIndex.from_product(
                [[store], [dept], history_dates], names=["Store", "Dept", "Date"]
            )
            history_matrix[index] = (
                lookup.reindex(history_index).fillna(0.0).to_numpy(np.float32)
            )
            lag_dates = forecast_dates - pd.Timedelta(weeks=self.seasonal_period)
            lag_index = pd.MultiIndex.from_product(
                [[store], [dept], lag_dates], names=["Store", "Dept", "Date"]
            )
            seasonal[index] = lookup.reindex(lag_index).fillna(0.0).to_numpy(np.float32)
        return (
            keys,
            history_dates,
            forecast_dates,
            history_matrix,
            np.clip(seasonal, self.clip_min, self.clip_max),
        )

    def _build_covariates(self, keys, history_dates, forecast_dates):
        dates = history_dates.append(forecast_dates)
        stores = sorted({int(store) for store, _ in keys})
        base = pd.MultiIndex.from_product(
            [stores, dates], names=["Store", "Date"]
        ).to_frame(index=False)
        frame = base.merge(
            self.external_features,
            on=["Store", "Date"],
            how="left",
            suffixes=("", "_feature"),
        ).sort_values(["Store", "Date"])
        holiday_column = (
            "IsHoliday_feature" if "IsHoliday_feature" in frame else "IsHoliday"
        )
        frame["holiday"] = frame[holiday_column].fillna(False).astype(bool)
        context_mask = frame["Date"].isin(history_dates)
        feature_columns = ["Temperature", "Fuel_Price", "CPI", "Unemployment"]
        context_medians = {
            name: pd.to_numeric(frame.loc[context_mask, name], errors="coerce").median()
            for name in feature_columns
        }
        for name in feature_columns:
            values = pd.to_numeric(frame[name], errors="coerce")
            frame[f"{name}_missing"] = values.isna().astype(float)
            frame[name] = (
                values.groupby(frame["Store"])
                .ffill()
                .fillna(context_medians[name])
                .fillna(0.0)
            )
        markdown_columns = [f"MarkDown{i}" for i in range(1, 6)]
        markdown = frame[markdown_columns].apply(pd.to_numeric, errors="coerce")
        frame["markdown_missing_count"] = markdown.isna().sum(axis=1).astype(float)
        frame["markdown_log_total"] = np.log1p(
            markdown.fillna(0.0).clip(lower=0.0).sum(axis=1)
        )
        iso = frame["Date"].dt.isocalendar()
        frame["week_sin"] = np.sin(2 * np.pi * iso.week.astype(float) / 52.0)
        frame["week_cos"] = np.cos(2 * np.pi * iso.week.astype(float) / 52.0)
        frame["month_sin"] = np.sin(2 * np.pi * frame["Date"].dt.month / 12.0)
        frame["month_cos"] = np.cos(2 * np.pi * frame["Date"].dt.month / 12.0)
        frame["event"] = [
            self._event_name(date, holiday)
            for date, holiday in zip(frame["Date"], frame["holiday"])
        ]
        frame["week_of_year"] = iso.week.astype(str)
        frame["holiday_cat"] = frame["holiday"].astype(int).astype(str)
        indexed = {
            (int(store), date): row
            for (store, date), row in frame.set_index(["Store", "Date"]).iterrows()
        }
        numerical = (
            feature_columns
            + [f"{name}_missing" for name in feature_columns]
            + [
                "markdown_log_total",
                "markdown_missing_count",
                "week_sin",
                "week_cos",
                "month_sin",
                "month_cos",
            ]
        )
        categorical = ["holiday_cat", "event", "week_of_year"]
        dynamic_numerical = {name: [] for name in numerical}
        dynamic_categorical = {name: [] for name in categorical}
        for store, _ in keys:
            rows = [indexed[(int(store), date)] for date in dates]
            for name in numerical:
                dynamic_numerical[name].append(
                    np.asarray([row[name] for row in rows], dtype=np.float32)
                )
            for name in categorical:
                dynamic_categorical[name].append([str(row[name]) for row in rows])
        store_metadata = self.stores.set_index("Store")
        static_numerical = {
            "Size": [float(store_metadata.loc[store, "Size"]) for store, _ in keys]
        }
        static_categorical = {
            "Store": [str(store) for store, _ in keys],
            "Dept": [str(dept) for _, dept in keys],
            "Type": [str(store_metadata.loc[store, "Type"]) for store, _ in keys],
        }
        return (
            dynamic_numerical,
            dynamic_categorical,
            static_numerical,
            static_categorical,
        )

    def predict(self, raw_test: pd.DataFrame) -> np.ndarray:
        missing = set(self.required_columns) - set(raw_test.columns)
        if missing:
            raise ValueError(f"raw test is missing columns: {sorted(missing)}")
        frame = (
            raw_test[list(self.required_columns)]
            .copy()
            .reset_index(names="__input_order")
        )
        frame["Date"] = pd.to_datetime(frame["Date"])
        keys, history_dates, forecast_dates, history_matrix, seasonal = (
            self._series_matrices(frame)
        )
        horizon = len(forecast_dates)
        model = self._load_model()
        raw_forecast, _ = model.forecast(
            horizon=horizon, inputs=[row for row in history_matrix]
        )
        raw_forecast = np.asarray(raw_forecast, float)
        if raw_forecast.ndim == 3:
            raw_forecast = raw_forecast[..., 0]
        raw_forecast = raw_forecast[:, -horizon:]
        raw_forecast = np.clip(
            np.nan_to_num(
                raw_forecast,
                nan=0.0,
                posinf=self.clip_max,
                neginf=0.0,
            ),
            self.clip_min,
            self.clip_max,
        )
        residual_history = (
            history_matrix[:, self.seasonal_period :]
            - history_matrix[:, : -self.seasonal_period]
        )
        residual_forecast, _ = model.forecast(
            horizon=horizon, inputs=[row for row in residual_history]
        )
        residual_forecast = np.asarray(residual_forecast, float)
        if residual_forecast.ndim == 3:
            residual_forecast = residual_forecast[..., 0]
        residual_forecast = residual_forecast[:, -horizon:]
        residual_reconstructed = np.clip(
            seasonal + np.nan_to_num(residual_forecast, nan=0.0),
            self.clip_min,
            self.clip_max,
        )
        dynamic_numerical, dynamic_categorical, static_numerical, static_categorical = (
            self._build_covariates(keys, history_dates, forecast_dates)
        )
        xreg_forecast, _ = model.forecast_with_covariates(
            inputs=[row for row in history_matrix],
            dynamic_numerical_covariates=dynamic_numerical,
            dynamic_categorical_covariates=dynamic_categorical,
            static_numerical_covariates=static_numerical,
            static_categorical_covariates=static_categorical,
            xreg_mode=self.xreg_mode,
        )
        xreg_forecast = np.asarray(xreg_forecast, float)
        if xreg_forecast.ndim == 3:
            xreg_forecast = xreg_forecast[..., 0]
        xreg_forecast = xreg_forecast[:, -horizon:]
        xreg_forecast = np.clip(
            np.nan_to_num(
                xreg_forecast,
                nan=0.0,
                posinf=self.clip_max,
                neginf=0.0,
            ),
            self.clip_min,
            self.clip_max,
        )
        expected_shape = (len(keys), horizon)
        forecast_shapes = {
            "seasonal": seasonal.shape,
            "raw": raw_forecast.shape,
            "residual": residual_reconstructed.shape,
            "xreg": xreg_forecast.shape,
        }
        if any(shape != expected_shape for shape in forecast_shapes.values()):
            raise ValueError(
                f"forecast shape contract failed: expected {expected_shape}, got {forecast_shapes}"
            )
        blended = np.clip(
            self.weights["seasonal"] * seasonal
            + self.weights["raw"] * raw_forecast
            + self.weights["residual"] * residual_reconstructed
            + self.weights["xreg"] * xreg_forecast,
            self.clip_min,
            self.clip_max,
        )
        key_index = {key: index for index, key in enumerate(keys)}
        horizon_index = {date: index for index, date in enumerate(forecast_dates)}
        predictions = np.asarray(
            [
                blended[
                    key_index[(int(row.Store), int(row.Dept))], horizon_index[row.Date]
                ]
                for row in frame.itertuples(index=False)
            ],
            dtype=np.float64,
        )
        if len(predictions) != len(raw_test) or not np.isfinite(predictions).all():
            raise ValueError("pipeline produced invalid predictions")
        return predictions

## instantiate and run the full raw-test contract

In [54]:
pipeline = TimesFMRawPipeline(
    history=train_raw,
    external_features=features_raw,
    stores=stores_raw,
    model_id=config["model_id"],
    weights=config["weights"],
    xreg_mode=config["xreg_mode"],
    seasonal_period=config["seasonal_period"],
    clip_min=config["clip_min"],
    clip_max=config["clip_max"],
    batch_size=config["batch_size"],
)
contract_started = time.time()
contract_predictions = pipeline.predict(test_raw)
contract_minutes = (time.time() - contract_started) / 60
prediction_hash = hashlib.sha256(contract_predictions.tobytes()).hexdigest()
print(
    {
        "pipeline_type": type(pipeline).__name__,
        "rows": len(contract_predictions),
        "minimum": float(contract_predictions.min()),
        "mean": float(contract_predictions.mean()),
        "maximum": float(contract_predictions.max()),
        "contract_minutes": contract_minutes,
        "sha256": prediction_hash,
    }
)
print(pipeline.metadata())

{'pipeline_type': 'TimesFMRawPipeline', 'rows': 115064, 'minimum': 0.0, 'mean': 16563.895239800087, 'maximum': 293424.17900390626, 'contract_minutes': 1.4629263917605082, 'sha256': '97529caaa57ed8334c3eaf7e9cdf1f7a95d0bb4efee4c09166433bf02a9dfefe'}
{'pipeline_type': 'TimesFMRawPipeline', 'model_id': 'google/timesfm-2.5-200m-pytorch', 'weights': {'seasonal': 0.4, 'raw': 0.05, 'residual': 0.45, 'xreg': 0.1}, 'xreg_mode': 'timesfm + xreg', 'seasonal_period': 52, 'stored_history_rows': 421570, 'stored_feature_rows': 8190, 'raw_input_columns': ['Store', 'Dept', 'Date', 'IsHoliday']}


## serialize, reload, and register the champion pipeline

In [55]:
pipeline_path = output_dir / "walmart_timesfm_v3_raw_pipeline.pkl"
manifest_path = output_dir / "pipeline_manifest.json"
reference_path = output_dir / "wandb_registry_reference.json"
pipeline._model = None
with pipeline_path.open("wb") as file:
    cloudpickle.dump(pipeline, file)
with pipeline_path.open("rb") as file:
    reloaded_pipeline = cloudpickle.load(file)
assert reloaded_pipeline.metadata() == pipeline.metadata()
manifest = {
    **pipeline.metadata(),
    "source_validation_wmae": 1588.8029448973086,
    "source_version": "v3",
    "contract_test_rows": len(test_raw),
    "contract_prediction_sha256": prediction_hash,
    "contract_minutes": contract_minutes,
    "pipeline_bytes": pipeline_path.stat().st_size,
    "dependencies": {"timesfm": "2.0.2", "model_checkpoint": config["model_id"]},
}
manifest_path.write_text(json.dumps(manifest, indent=2))
registry_reference = {
    "registry_uri": f"{config['registry_target']}:champion",
    "registry_target": config["registry_target"],
    "alias": "champion",
    "pipeline_file": pipeline_path.name,
}
reference_path.write_text(json.dumps(registry_reference, indent=2))
print(
    {
        "pipeline_path": str(pipeline_path),
        "pipeline_mb": pipeline_path.stat().st_size / 1024**2,
        **registry_reference,
    }
)

{'pipeline_path': '/content/artifacts/timesfm_final_pipeline/walmart_timesfm_v3_raw_pipeline.pkl', 'pipeline_mb': 13.9843111038208, 'registry_uri': 'wandb-registry-model/Walmart_TimesFM_Raw_Pipeline:champion', 'registry_target': 'wandb-registry-model/Walmart_TimesFM_Raw_Pipeline', 'alias': 'champion', 'pipeline_file': 'walmart_timesfm_v3_raw_pipeline.pkl'}


In [56]:
run = wandb.init(
    entity=config["wandb_entity"],
    project=config["wandb_project"],
    group="timesfm-finalization",
    job_type="model-registration",
    name=config["wandb_run_name"],
    config=config,
)
source_artifact = run.use_artifact(config["source_evaluation_artifact"])
run.log(
    {
        "pipeline/raw_input_contract_passed": 1,
        "pipeline/test_rows_verified": len(test_raw),
        "pipeline/contract_minutes": contract_minutes,
        "pipeline/prediction_min": float(contract_predictions.min()),
        "pipeline/prediction_mean": float(contract_predictions.mean()),
        "pipeline/prediction_max": float(contract_predictions.max()),
        "pipeline/serialized_mb": pipeline_path.stat().st_size / 1024**2,
        "validation/champion_wmae": manifest["source_validation_wmae"],
    }
)
model_artifact = wandb.Artifact(
    name=config["artifact_name"],
    type="model",
    description="complete timesfm v3 champion pipeline accepting raw walmart test.csv rows",
    metadata=manifest,
)
for path in (pipeline_path, manifest_path, reference_path):
    model_artifact.add_file(str(path))
logged_artifact = run.log_artifact(model_artifact, aliases=["champion", "latest"])
logged_artifact.wait()
run.link_artifact(
    artifact=logged_artifact,
    target_path=config["registry_target"],
    aliases=["champion", "latest"],
)
run.summary.update(
    {
        **manifest,
        "registry/target": config["registry_target"],
        "registry/alias": "champion",
        "source_evaluation_artifact": source_artifact.name,
    }
)
run.finish()
print(
    {"registered": True, **registry_reference, "source_artifact": source_artifact.name}
)

pipeline/contract_minutes,▁
pipeline/prediction_max,▁
pipeline/prediction_mean,▁
pipeline/prediction_min,▁
pipeline/raw_input_contract_passed,▁
pipeline/serialized_mb,▁
pipeline/test_rows_verified,▁
validation/champion_wmae,▁
contract_minutes,1.46293
contract_prediction_sha256,97529caaa57ed8334c3e...
contract_test_rows,115064


{'registered': True, 'registry_uri': 'wandb-registry-model/Walmart_TimesFM_Raw_Pipeline:champion', 'registry_target': 'wandb-registry-model/Walmart_TimesFM_Raw_Pipeline', 'alias': 'champion', 'pipeline_file': 'walmart_timesfm_v3_raw_pipeline.pkl', 'source_artifact': 'timesfm-v3-xreg-corrected-calibration:latest'}


## completion criteria

the pipeline is ready for inference only after the final cell prints `registered: true`. the inference notebook must install the pinned dependencies, download `wandb-registry-model/Walmart_TimesFM_Raw_Pipeline:champion`, load the pickle and call `pipeline.predict(raw_test)` without rebuilding features outside the pipeline.